In [2]:
import sqlite3
import pandas as pd 

conn = sqlite3.connect("../data/olist.db")

def q(sql):
    return pd.read_sql(sql, conn)

q("""
SELECT COUNT(*) AS orders,
COUNT (DISTINCT customer_id) AS customers
FROM orders
""")

,orders,customers
0,99441,99441


Выручка = сумма price по заказам со статусом delivered (без стоимости доставки freight_value). Клиент - уникальный customer_unique_id , а не customer_id. customer_id создаётся заново для каждого заказа.  

In [3]:
q("""
SELECT COUNT(*) AS orders_count,
       COUNT(DISTINCT customer_id) AS customer_ids,
       (SELECT COUNT(DISTINCT customer_unique_id) FROM customers) AS unique_customers_count
FROM orders;
""")

,orders_count,customer_ids,unique_customers_count
0,99441,99441,96096


In [6]:
q("""
SELECT MIN(order_purchase_timestamp) AS first_order,
       MAX(order_purchase_timestamp) AS last_order
FROM orders;
""")

,first_order,last_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


In [ ]:
q("""
SELECT order_status,
       COUNT(*) AS orders_count,
       ROUND(COUNT(*)/(SELECT COUNT(*) FROM orders), 2) as percent
FROM orders
GROUP BY order_status
ORDER BY orders_count DESC;
""")

Подавляющее большинство заказов в статусе delivered.
Далее весь анализ делается по доставленным заказам

In [ ]:
q("""
SELECT strftime('%Y-%m', orders.order_purchase_timestamp) AS month,
       COUNT(DISTINCT orders.order_id) AS orders,
       ROUND(SUM(order_items.price),2) AS revenue
FROM orders
JOIN order_items ON orders.order_id = order_items.order_id
WHERE orders.order_status = 'delivered'
GROUP BY month
ORDER BY month;
""")

Наибольшая выручка в ноябре 2017. Месяцы с малым числом заказов (конец 2016 года и сентябрь-октябрь 2018 года) неполные

In [ ]:
q("""
SELECT payment_type,
       COUNT(DISTINCT order_id) AS orders,
       ROUND(SUM(payment_value),2) AS total_paid
FROM order_payments
GROUP BY payment_type
ORDER BY total_paid DESC;
""")

In [ ]:
q("""
SELECT ROUND(SUM(oi.price)/COUNT(DISTINCT oi.order_id)) AS avg_order_value
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered';
""")

Средний чек = 137

Какие ТОП-10 категорий по выручке?

In [ ]:
q("""
SELECT COALESCE(t.product_category_name, t.product_category_name_english) AS category,
       COUNT(DISTINCT oi.order_id) AS orders,
       ROUND(SUM(oi.price), 2) AS revenue
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
LEFT JOIN product_category_name_translation t 
       ON p.product_category_name = t.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY category
ORDER BY revenue DESC
LIMIT 10;
""")

Какие ТОП-10 штатов клиентов по выручке?

In [ ]:
q("""
SELECT c.customer_state AS state,
       COUNT(DISTINCT oi.order_id) AS orders,
       ROUND(SUM(oi.price), 2) AS revenue
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY state
ORDER BY revenue DESC
LIMIT 10;
""")